# LLM-Assisted Feature Engineering
This notebook documents the use of AI asistent (LLaMA 3.1 via Groq) to suggest the features for our Logistic Regressiong Model

## 1. Query the LLM

In [1]:
import os
from groq import Groq

# To set up: add your key here and uncomment the below code:
# GROQ_API_KEY = addyourkey
client = Groq(api_key="GROQ_API_KEY")


prompt_content = """
You are a Medical AI Feature Engineering Assistant. We are building a Logistic Regression model to predict Heart Disease using a pre-processed version of the UCI Heart Disease dataset.

CRITICAL ARCHITECTURE CONSTRAINT:
All our columns have ALREADY been fully standardized/scaled using a StandardScaler (Z-score transformed to Mean=0, Std=1). This means binary columns and continuous columns alike are represented as standard deviations away from the mean (containing negative numbers and fractions). 

Our exact 15 feature column names are:
1.  Age (Standardized continuous)
2.  Sex (Standardized binary)
3.  RestingBP (Standardized continuous)
4.  Cholesterol (Standardized continuous)
5.  FastingBS (Standardized binary)
6.  MaxHR (Standardized continuous)
7.  ExerciseAngina (Standardized binary)
8.  Oldpeak (Standardized continuous)
9.  ChestPainType_ATA (Standardized one-hot)
10. ChestPainType_NAP (Standardized one-hot)
11. ChestPainType_TA (Standardized one-hot)
12. RestingECG_Normal (Standardized one-hot)
13. RestingECG_ST (Standardized one-hot)
14. ST_Slope_Flat (Standardized one-hot)
15. ST_Slope_Up (Standardized one-hot)

Target Variable: HeartDisease (0 = Normal, 1 = Heart Disease)

Please suggest 3 clinically relevant engineered features or interaction terms that a linear classifier like Logistic Regression can exploit. Because the features are already Z-score scaled, simple division/multiplication may distort the distribution. Please specify if we should combine these features using addition/subtraction, or how to handle the sign of the interaction terms mathematically. Keep explanations simple and explain the clinical intuition.
"""

print("Querying the LLM for Feature Engineering Suggestions...")

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant", 
    messages=[
        {"role": "user", "content": prompt_content}
    ]
)

print("\n=== LLM SUGGESTIONS ===")
print(completion.choices[0].message.content)

Querying the LLM for Feature Engineering Suggestions...

=== LLM SUGGESTIONS ===
Given the pre-processed continuous and binary features, I'd suggest three clinically relevant engineered features or interaction terms to improve the model. Each suggestion is motivated by a relevant clinical concept.

**1. Age-Risk Interaction (Interaction Term):**

Feature: `Age * Risk`

Clinically Relevant Concept: Age is a well-known risk factor for heart disease. The interaction term represents the combined effect of age and risk on heart disease.

To implement this interaction term, multiply `Age` by `Risk` (i.e., combine the standardized binary features `Sex`, `FastingBS`, `ExerciseAngina`, `ChestPainType_ATA`, `ChestPainType_NAP`, `ChestPainType_TA`, `RestingECG_Normal`, `RestingECG_ST`, `ST_Slope_Flat`, and `ST_Slope_Up`). Use the following formula to handle the sign of the interaction term:

`Age * Risk = Age * (sum of 1 for present features)`

The interaction term will be positive if `Sex` is 1,

## 2. Evaluation

### Sugestion 1 - AgexRisk score  
Clinical view: Age combined with multiple indicators represent compounded risk  
Evaluation: **rejected**  
Multiply a continuouse Z-score (Age) by a sum of standardized binary feature creates an unstable surface (When Age is negative x Risk is positive --> product became negative --> model see that young high-risk patient as lower risk than a young low-risk patient --> reducing interpretability and potentially Misleading

### Suggestion 2 - BP-Cholesterol Ratio
Clinical view: A high log(RestingBP/Cholesterol) may indicate cardiovascular strain  
Evaluation: **rejected**  
Both RestingBP and Cholesterol are Z-score standardize --> when the demnominator approches zero/ negative value --> take a log of negative number is undefined. --> cause error

### Suggestion 3 - Exercise Stress Index
Clinical view: Combines ST depression (oldpeak) with exercise-induced and ST slope --> a diagnostic tool
Evaluation: **Selected for testing**  
- Multiply two standardize binary features --> less harmful than multiplying two continous Z-scores.
- The domination operation here is addition.